# Day 4 v2 — Notebook 11: Ridge Stacking Ensemble

Combine val predictions from all trained models via Ridge regression.
Target: MAE < 65k VND (P0), < 60k VND (stretch).

## Required val_predictions files

| File | Source | Individual Test MAE |
|------|--------|--------------------|
| `val_predictions/dnn_tfidf_val.json` | NB01 | 77.3k |
| `val_predictions/dnn_tfidf_test.json` | NB01 | |
| `val_predictions/dnn_hashvec_val.json` | NB02 | 80.0k |
| `val_predictions/dnn_hashvec_test.json` | NB02 | |
| `val_predictions/aitvn_val.json` | NB05 (frozen) | 76.1k |
| `val_predictions/aitvn_test.json` | NB05 (frozen) | |
| `val_predictions/aitvn_finetune_val.json` | NB06 (top-4) | 74.2k |
| `val_predictions/aitvn_finetune_test.json` | NB06 (top-4) | |
| `val_predictions/aitvn_improved_val.json` | NB09 (top-8+RDrop) | pending |
| `val_predictions/aitvn_improved_test.json` | NB09 (top-8+RDrop) | |
| `val_predictions/phobert_base_improved_val.json` | NB10 (top-8+RDrop) | 77.6k |
| `val_predictions/phobert_base_improved_test.json` | NB10 (top-8+RDrop) | |

Missing files are skipped automatically — ensemble uses whatever is available.

In [ ]:
import json
import sys
import numpy as np
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sys.path.append("..")
from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate

## 1. Config

In [ ]:
PRED_DIR = Path("val_predictions")
EVAL_SIZE = 200  # evaluator.py uses first 200 test samples

# All candidate models: (display_name, val_json, test_json)
MODEL_CONFIGS = [
    ("DNN+TF-IDF",       "dnn_tfidf_val.json",              "dnn_tfidf_test.json"),
    ("DNN+HashVec",      "dnn_hashvec_val.json",            "dnn_hashvec_test.json"),
    ("AITeamVN-frozen",  "aitvn_val.json",                  "aitvn_test.json"),
    ("AITeamVN-ft4",     "aitvn_finetune_val.json",         "aitvn_finetune_test.json"),
    ("AITeamVN-top8",    "aitvn_improved_val.json",         "aitvn_improved_test.json"),
    ("PhoBERT-top8",     "phobert_base_improved_val.json",  "phobert_base_improved_test.json"),
]

## 2. Load data

In [ ]:
print("Loading items_tv_v9...")
train_items, val_items, test_items = Item.from_hub("SeanSunny/items_tv_v9")

y_val  = np.array([item.price for item in val_items],  dtype=np.float32)   # 3926
y_test = np.array([item.price for item in test_items], dtype=np.float32)   # 3872

print(f"val: {len(y_val)} | test: {len(y_test)}")

## 3. Load predictions (skip missing)

In [ ]:
loaded_names = []
val_preds_list  = []
test_preds_list = []

for name, val_file, test_file in MODEL_CONFIGS:
    vp = PRED_DIR / val_file
    tp = PRED_DIR / test_file
    if not vp.exists() or not tp.exists():
        print(f"SKIP {name}: {val_file} not found")
        continue
    v_preds = np.array(json.loads(vp.read_text()), dtype=np.float32)
    t_preds = np.array(json.loads(tp.read_text()), dtype=np.float32)
    if len(v_preds) != len(y_val) or len(t_preds) != len(y_test):
        print(f"SKIP {name}: shape mismatch (val={len(v_preds)}, test={len(t_preds)})")
        continue
    loaded_names.append(name)
    val_preds_list.append(v_preds)
    test_preds_list.append(t_preds)
    print(f"OK   {name}")

print(f"\nLoaded {len(loaded_names)} models: {loaded_names}")
assert len(loaded_names) >= 2, "Need at least 2 models to stack"

X_val  = np.stack(val_preds_list,  axis=1)  # (3926, n_models)
X_test = np.stack(test_preds_list, axis=1)  # (3872, n_models)

## 4. Per-model MAE on val set

In [ ]:
print(f"{'Model':<20} {'Val MAE':>10} {'Val R2':>8}")
print("-" * 42)
for name, preds in zip(loaded_names, val_preds_list):
    mae = mean_absolute_error(y_val, preds)
    r2  = r2_score(y_val, preds) * 100
    print(f"{name:<20} {mae:>8.1f}k  {r2:>6.1f}%")

## 5. Ridge stacking — alpha grid search

In [ ]:
ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

best_alpha, best_mae, best_ridge = None, float("inf"), None

print(f"{'Alpha':>10} {'Val MAE':>10}")
print("-" * 24)
for alpha in ALPHAS:
    ridge = Ridge(alpha=alpha, fit_intercept=True)
    ridge.fit(X_val, y_val)
    val_mae = mean_absolute_error(y_val, ridge.predict(X_val))
    print(f"{alpha:>10.2f} {val_mae:>8.1f}k")
    if val_mae < best_mae:
        best_mae, best_alpha, best_ridge = val_mae, alpha, ridge

print(f"\nBest alpha={best_alpha} | Val MAE={best_mae:.1f}k")

## 6. Stacking weights

In [ ]:
coefs = best_ridge.coef_
intercept = best_ridge.intercept_

print(f"{'Model':<20} {'Coef':>10}")
print("-" * 34)
for name, c in zip(loaded_names, coefs):
    print(f"{name:<20} {c:>10.4f}")
print(f"{'intercept':<20} {intercept:>10.4f}")

## 7. Full val set evaluation

In [ ]:
y_val_pred = best_ridge.predict(X_val)
val_mae    = mean_absolute_error(y_val, y_val_pred)
val_mse    = mean_squared_error(y_val, y_val_pred)
val_r2     = r2_score(y_val, y_val_pred) * 100

print("=" * 40)
print(f"Ensemble val MAE : {val_mae:.1f}k VND")
print(f"Ensemble val MSE : {val_mse:,.0f}")
print(f"Ensemble val R2  : {val_r2:.1f}%")
print("=" * 40)

## 8. Test set evaluation — 200 samples (evaluator.py)

In [ ]:
y_test_pred = best_ridge.predict(X_test)  # 3872 samples
test_pred_200 = y_test_pred[:EVAL_SIZE]

# Quick numpy eval on full test
test_mae_full = mean_absolute_error(y_test, y_test_pred)
test_r2_full  = r2_score(y_test, y_test_pred) * 100
print(f"Full test (3872) MAE={test_mae_full:.1f}k  R2={test_r2_full:.1f}%")
print()

# Evaluator on first 200 (consistent with all other notebooks)
idx = 0
def ensemble_predictor(item):
    global idx
    pred = test_pred_200[idx]
    idx += 1
    return float(pred)

results = evaluate(ensemble_predictor, test_items, size=EVAL_SIZE)
print(f"\nEnsemble test MAE (200): {results['mae']:.1f}k VND")
print(f"Ensemble test R2  (200): {results['r2']:.1f}%")

## 9. Summary

In [ ]:
print("=" * 50)
print("DAY 4 V2 — ENSEMBLE RESULTS")
print("=" * 50)
print(f"Models stacked : {len(loaded_names)}")
print(f"  " + ", ".join(loaded_names))
print(f"Best alpha     : {best_alpha}")
print()
print(f"Val MAE        : {val_mae:.1f}k VND")
print(f"Val R2         : {val_r2:.1f}%")
print(f"Test MAE (3872): {test_mae_full:.1f}k VND")
print(f"Test MAE  (200): {results['mae']:.1f}k VND")
print(f"Test R2   (200): {results['r2']:.1f}%")
print()
target_p0 = results['mae'] < 65.0
target_stretch = results['mae'] < 60.0
print(f"Target < 65k   : {'PASS' if target_p0 else 'FAIL'}")
print(f"Target < 60k   : {'PASS' if target_stretch else 'FAIL'}")
print("=" * 50)